# Práctica 4 - Representación

La tarea va a consistir en el estudio completo de un conjunto de datos de enfermedades cardiovasculares.

Primero, se va a realizar un estudio exhaustivo de los datos, analizando las distribuciones de las variables y la relación entre estas, para determinar qué atributos son más importantes para predecir el objetivo final. Este análisis irá principalmente fundamentado por la visualización de los datos, esto es, gráficas que permitirán observar visualmente los patrones encontrados. Una vez conocido y estudiado el conjunto de datos con el que se va a trabajar, pasaremos a aplicar técnicas de aprendizaje automático para diseñar modelos que permitan predecir el riesgo de padecer enfermedades cardiovasculares. En esta sección también serán importantes las técnicas de clustering y de detección de anomalías, de tal manera que no se distorsionen los resultados finales.

Realizar un estudio completo de un dataset, haciendo especial hincapié en la visualización de los datos, tratando de mejorar las gráficas estudiadas hasta el momento. Además, también se va a prestar especial atención a técnicas de clustering y se propondrán modelos adicionales para predecir el resultado final.

- Preparación de datos (visualización de variables, detección de outliers)
- Relación entre atributos (tablas de contingencia, matriz de correlaciones)
- Técnicas de clustering (K-Means)
- Técnicas de reducción de la dimensionalidad (PCA)
- Modelos de predicción (Regresión Logística)

#### Conjunto de datos

El conjunto de datos de enfermedades cardiovasculares es un dataset de gran tamaño que contiene información sobre 70000 pacientes y su estado de salud cardiovascular. Este conjunto de datos incluye 11 características diferentes que se han recopilado sobre cada paciente, como la edad, el género, la presión arterial, el colesterol y el tabaquismo, entre otras. El objetivo de este conjunto de datos es predecir si un paciente tiene una enfermedad cardiovascular o no, lo que lo convierte en un problema de clasificación binaria.

Los atributos del dataset son:

- id : identificador de cada paciente
- age: edad (en días) de cada paciente
- gender: género (1: mujer, 2:hombre)
- height: altura en cm
- weight: peso en kg
- ap_hi: tensión arterial sistólica
- ap_lo: tensión arterial distólica
- cholesterol: colesterol del paciente (1: normal, 2: elevado, 3: exageradamente elevado)
- gluc: glucosa (1: normal, 2: elevado, 3: exageradamente elevado)
- smoke: indica si el paciente fuma o no (0: no, 1: sí)
- alco: indica si el paciente bebe alcohol o no (0: no, 1: sí)
- active: indica si el paciente realiza actividad física (0: no, 1: sí)
- **cardio: variable de salida/objetivo. Presencia o ausencia de enfermedad cardiovascular**

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession \
    .builder \
    .master("local[*]") \
    .appName("Ejemplo pySparkSQL") \
    .config("spark.sql.warehouse.dir", "file:///D:/tmp/spark-warehouse") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
# Lectura de datos.
# Delimitador ; y con header.
df = #RELLENAR

# Cacheamos el df
df.#RELLENAR

En primer lugar, se quiere visualizar el conjunto de datos y 15 instancias del mismo, de tal forma que se entienda bien el dataset con el que se va a trabajar.

In [ ]:
df.#RELLENAR

### Análisis de Datos

Esta sección va a estar destinada a realizar un análisis profundo de los datos. Para empezar, simplemente estudiaremos los tipos de datos de las variables y realizaremos las modificaciones necesarias para poder trabajar con ellas. Una vez tengamos el conjunto de datos completo, pasaremos a visualizar las relaciones entre los atributos.

In [ ]:
# Tipos de datos de las variables
df.#RELLENAR

Se observa que todas las variables se han definido como *string*, cuando la edad, altura, peso y la presión arterial deben de ser *integer*.

In [ ]:
from pyspark.sql.functions import col
from test_helper import Test

#Lista variables numéricas
lista_num = ["age", "height", "weight", "ap_hi", "ap_lo"]

# Convertir las columnas necesarias a enteros utilizando el método cast()
df = df.select([col(column).cast("int").alias(column) if column in lista_num else col(column) for column in df.columns])

# Comprobar que los tipos de datos se han actualizado correctamente
print(df.#RELLENAR)

In [ ]:
Test.assertEquals(df.dtypes, [('id', 'string'), ('age', 'int'), ('gender', 'string'), ('height', 'int'), ('weight', 'int'), ('ap_hi', 'int'), ('ap_lo', 'int'), ('cholesterol', 'string'), ('gluc', 'string'), ('smoke', 'string'), ('alco', 'string'), ('active', 'string'), ('cardio', 'string')],
                 'Tipos de datos incorrecto')

Para las variables numéricas, se quiere hacer un resumen estadístico, incluyendo la media, desviación estándar, mínimo y máximo. Para las variables categóricas, se quiere conocer el número de pacientes que hay para cada categoría.

In [ ]:
# Variables numéricas
df_num = df.select(*[col(c) for c in df.columns if c in lista_num])

# Variables categóricas (eliminar id porque no nos interesa para este análisis inicial)
df_cat = df.select(*[c for c in df.columns if c not in lista_num]).#RELLENAR

In [ ]:
from pyspark.sql.functions import format_number

# Resumen estadístico
summary = df_num.describe().select(*['summary'] + [format_number(col(c).cast("double"), 2).alias(c) for c in df_num.columns if c not in ['id', 'gender']]).toPandas().set_index('summary').transpose()
summary

La edad está en días. En general, estamos más acostumbrados a trabajar en años, por lo que la convertiremos en años. Me interesa la edad aproximada, luego nos quedamos con el número entero y no con decimales.

In [ ]:
from pyspark.sql.functions import floor

df_num = df_num.withColumn("age", floor(df_num["age"] / 365))

# También para el dataset completo
df = df.withColumn("age", floor(df["age"] / 365))

In [ ]:
df_num.show(3)

Calcular la media de edad del dataset.

In [ ]:
media_edad = #RELLENAR
print("La media de edad es: ", media_edad)

In [ ]:
Test.assertEquals(media_edad, 52.840671428571426, 'Conversión incorrecta')

In [ ]:
# Conteo atributo categórico
for col in df_cat.columns:
    # Agrupar por categoría y contar las instancias
    #RELLENAR

Con esta información previa inicial, queremos saber ahora si existen datos faltantes.

In [ ]:
from pyspark.sql.functions import count, when, col

# Contar la cantidad de valores faltantes en cada columna
missing_values_count = df.select([count(when(col(c).#RELLENAR, c)).alias(c) for c in df.columns]).show()

Observamos que no hay datos faltantes. De este modo, podemos empezar ya con el estudio de los datos.

#### 1. Detección de outliers

Un valor atípico es un punto de datos que difiere significativamente de otras observaciones. Estos valores anómalos se van a estudiar únicamente sobre variables numéricas, ya que las categóricas únicamente recogen las categorías definidas. Hay varias formas de detectar valores atípicos:

- Diagrama de cajas
- Uso del rango intercuartílico IQR
- Mediante el diagrama de dispersión


Cada variable abarca un rango diferente. He agrupado por variables con rangos similares para realizar los gráficos.

In [ ]:
import matplotlib.pyplot as plt

# Crear una figura
fig, ax = plt.subplots()

# Crear los boxplots
boxplots = []
i = 0
for col in ["age", "weight", "height"]:
    bp = ax.boxplot(df_num.select(col).dropna().rdd.flatMap(lambda x: x).collect(), positions=[i+1], widths=0.6, showfliers=True)
    boxplots.append(bp)
    i = i+1

# Configurar el eje x
ax.set_xticks(range(1, len(["age", "weight", "height"])+1))
ax.set_xticklabels(["age", "weight", "height"])

# Agregar una leyenda
ax.legend([bp["boxes"][0] for bp in boxplots], ["age", "weight", "height"])

# Mostrar la figura
plt.title('Box Plot for Variables with Outliers')
plt.show()

Aunque veamos que hay datos que se encuentran por encima/debajo de los bigotes del diagrama de cajas, podemos observar que no son completamente anómalos. En el caso des peso, simplemente corresponden a individuos con un peso más elevado de lo habitual (o menos, si lo miramos por debajo). Lo mismo ocurre con la altura. Por tanto, no sería una buena práctica eliminarlos, ya que podrían ofrecer importantes conclusiones al estudio.

In [ ]:
# Crear una figura
fig, ax = plt.subplots()

# Crear los boxplots
boxplots = []
i = 0
for col in ["ap_hi", "ap_lo"]:
    bp = ax.boxplot(df_num.select(col).dropna().rdd.flatMap(lambda x: x).collect(), positions=[i+1], widths=0.6, showfliers=True)
    boxplots.append(bp)
    i = i+1

# Configurar el eje x
ax.set_xticks(range(1, len(["ap_hi", "ap_lo"])+1))
ax.set_xticklabels(["ap_hi", "ap_lo"])

# Agregar una leyenda
ax.legend([bp["boxes"][0] for bp in boxplots], ["ap_hi", "ap_lo"])

# Mostrar la figura
plt.title('Box Plot for Variables with Outliers')
plt.show()

En este caso, la presión arterial sistólica y distólica no es posible que sea tan elevada (en general, sobrepasar el valor de 140 mmHg ya supone un riesgo elevado). Por este motivo, no es posible que tome valores en el rango de miles. Se deberán eliminar las instancias siguiendo el siguiente proceso.

Calcula los límites de los valores atípicos utilizando el rango intercuartílico (IQR), que es la distancia entre el primer y tercer cuartil. Luego, identificar los valores que superan el umbral superior calculado y actualizar el dataframe.

In [ ]:
# Definir una función que establezca el umbral
def umbralOultiers(atributo):
    q1, q3 = df.approxQuantile(atributo, [0.25, 0.75], 0.005)
    iqr = q3 - q1
    umbral_superior = q3 + 1.5*iqr
    umbral_inferior = q3 - 1.5*iqr

    return umbral_superior, umbral_inferior

In [ ]:
# Establezco los umbrales
umbral_superior_ap_hi, umbral_inferior_ap_hi = umbralOultiers("ap_hi")
umbral_superior_ap_lo, umbral_inferior_ap_lo = umbralOultiers("ap_lo")

# Creo las listas de oultiers
outliers_ap_hi = [y for y in df.select("ap_hi").dropna().rdd.flatMap(lambda x: x).collect() if #RELLENAR]
outliers_ap_lo = [y for y in df.select("ap_lo").dropna().rdd.flatMap(lambda x: x).collect() if #RELLENAR]


print("Número de outliers detectados en ap_hi: ", len(outliers_ap_hi), "\nNúmero de outliers detectados en ap_hi: ", len(outliers_ap_lo))

In [ ]:
assert len(outliers_ap_hi) == 5173, "Test failed: outliers_ap_hi is wrong"
assert len(outliers_ap_lo) == 15184, "Test failed: outliers_ap_lo is wrong"

Estos valores anómalos los vamos a eliminar. Generalmente, no es la mejor opción, ya que se podrían reemplazar por algún valor estadístico, como por ejemplo, la media, mediana o moda. También se podrían transformar, mediante transformaciones logarítmicas o Box-Cox. Sin embargo, en esta ocasión, al tratarse de un conjunto de datos con numerosas instancias, eliminar los valores atípicos no va a suponer un cambio grande. Además, para este problema en concreto, los outliers en cuestión son datos que probablemente están mal recogidos, puesto que corresponden a valores imposibles para las variables, lo que distorsionaría los resultados finales.

La justificación de la eliminación de estos ejemplos reside en que el rango normal de la presión arterial sistólica es de 90 a 120 mmHg y el rango normal de la presión arterial diastólica es de 60 a 80 mmHg. En este caso, estabamos obteniendo valores en torno a 16000 mmHg o incluso -150 mmHg, claramente inviable.

In [ ]:
# El operador ~ se utiliza para negar la condición
df_clean = df.where(~(df["ap_hi"].isin(outliers_ap_hi)) & ~(df["ap_lo"].isin(outliers_ap_lo)))

Mostras las tres primeras filas del nuevo dataframe

In [ ]:
# Cacheamos el dataframe (es el que usaremos) y mostramos 3 filas
df_clean.cache()

# Mostramos 3 filas
df_clean.show(3)

In [ ]:
# Numero total de instancias
instancias = #RELLENAR

In [ ]:
Test.assertEquals(instancias, 53455, 'Dataset incorrectp')

1 test passed.


Ya hemos detectado los outliers, pero queremos visualizar también las distribuciones de esos atributos. Para ello, realizamos los siguientes histogramas con su respectiva curva de densidad.

In [ ]:
import seaborn as sns

def histogramaDensidad(atributo, subplot):
    # Histograma y densidad del atributo
    atributo_data = df_clean.select(atributo).#RELLENAR.rdd.flatMap(lambda x: x).collect()

    # Crear el subplot correspondiente
    plt.subplot(subplot)
    sns.histplot(atributo_data, kde=True, color="lightblue")

    # Configurar el título y los labels del subplot
    plt.title('Distribution '+str(atributo))
    plt.xlabel(str(atributo))
    plt.ylabel('Density')

# Crear la figura con dos filas y dos columnas
fig = plt.figure(figsize=(15,12))
plt.subplots_adjust(hspace=0.6)

# Generar los 5 gráficos
histogramaDensidad('height', 232)
histogramaDensidad('weight', 233)
histogramaDensidad('ap_hi', 234)
histogramaDensidad('ap_lo', 235)
histogramaDensidad('age', 231)

# Mostrar la figura con los 5 gráficos
plt.show()

Observamos que el conjunto de datos ya está preparado y ajustado al problema en cuestión. Los datos excesivamente elevados, que encontrabamos en los extremos, han desaparecido, para dar lugar a rangos habituales de la presión arterial sistólica y distólica.

Finalmente, vamos a analizar brevemente las frecuencias de las variables categóricas. Para ello, generaremos un gráfico de sectores agrupado.

In [ ]:
# Definir las columnas categóricas
categorias = #RELLENAR

# Definir los colores de los sectores
colors = ['#ff9999','#66b3ff','#99ff99','#ffcc99', '#ffb3e6', '#b3b3ff']

# Crear los subplots
fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(15, 8))

# Generar los gráficos de pastel para cada columna
for i, col in enumerate(categorias):
    # Obtener los datos para el gráfico
    no_cardio = [len(df.filter(#RELLENAR).filter(#RELLENAR).collect()) for cat in df.select(col).distinct().rdd.flatMap(lambda x: x).collect()]
    yes_cardio = [len(df.filter(#RELLENAR).filter(#RELLENAR).collect()) for cat in df.select(col).distinct().rdd.flatMap(lambda x: x).collect()]
    total_cardio = sum(yes_cardio)
    total_no_cardio = sum(no_cardio)
    total_personas = total_cardio + total_no_cardio

    # Configurar los datos para el gráfico de pastel
    labels = [f'{cat} - No Cardio' for cat in df.select(col).distinct().rdd.flatMap(lambda x: x).collect()] + [f'{cat} - Cardio' for cat in df.select(col).distinct().rdd.flatMap(lambda x: x).collect()]
    sizes = no_cardio + yes_cardio

    # Crear el gráfico de pastel en el lugar correspondiente
    row = i // 3
    colum = i % 3
    axs[row, colum].pie(sizes, colors=colors, labels=labels, autopct='%1.1f%%', startangle=90)

    # Configurar el título del gráfico
    axs[row, colum].set_title(f'Cardio by {col}')

# Mostrar los gráficos
plt.show()

De esta manera, podemos hacernos una idea de cómo actúa cada variable categórica frente a la variable respuesta *cardio*. Profundizaremos más en la siguiente sección.

#### 2. Estudiar la relación entre variables

Una vez conocidos los atributos del dataset, sus distribuciones e influencia sobre la variable objetivo *cardio*, pasaremos a analizar las relaciones existentes de estas variables. El objetivo de esta sección es quedarnos con las variables más importantes.

En primer lugar haremos un gráfico de dispersión, que representa la relación entre dos variables numéricas. Si los puntos tienden a agruparse alrededor de una línea recta, la relación entre las variables es lineal y su fuerza se puede medir a través del coeficiente de correlación. Si los puntos siguen una tendencia curva, la relación es no lineal. Si no hay tendencia aparente, no hay relación.

In [ ]:
# Selecionar las columnas numéricas
columns = #RELLENAR
numeric_df = df_clean.select(*columns)

# Crear un gráfico de dispersión para cada combinación de columnas
fig, axs = plt.subplots(nrows=len(columns), ncols=len(columns), figsize=(15,15))

for i, column_i in enumerate(columns):
    for j, column_j in enumerate(columns):
        axs[i, j].scatter(numeric_df.select(column_i).collect(), numeric_df.select(column_j).collect(), s=1)
        if j == 0:
            axs[i, j].set_ylabel(column_i)
        if i == len(columns) - 1:
            axs[i, j].set_xlabel(column_j)

plt.show()

Se aprecian bandas en ciertos gráficos, mientras que en otros simplemente observamos agrupaciones de individuos. En general, no se aprecia ninguna relación clara entre las variables. Si así lo fuera, quizás sería interesante suprimir alguna de estas por ser redundante. De todas formas, calculamos la matriz de correlación y la graficamos para asegurarnos.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# Seleccionar las columnas numéricas
columns = #RELLENAR
numeric_df = df.select(*columns)

# Convertir las columnas en un vector
assembler = #RELLENAR
numeric_df = #RELLENAR

# Calcular la matriz de correlación
correlation_matrix = #RELLENAR.head()

# Obtener la matriz de correlación como un array de numpy
corr_array = correlation_matrix[0].toArray()

# Mostrar la matriz de correlación como una matriz de colores
plt.imshow(corr_array, cmap='coolwarm')
plt.colorbar()
plt.xticks(range(len(columns)), columns, rotation=45)
plt.yticks(range(len(columns)), columns)
plt.show()

In [ ]:
# Mostrar la correlación
print(corr_array)

In [ ]:
import numpy as np
Test.assertEquals(np.mean(corr_array), 0.23229316827504307,'Matriz de correlación incorrecta')

Efectivamente, no hay atributos correlados, luego nos hacen falta todas las variables para posteriores predicciones. La línea diagonal central enfrenta a cada variable con ella misma.

Para estudiar las posibles relaciones entre variables categóricas, se analizarán tablas de contingencia y gráficos de mapas de calor.

In [ ]:
import pyspark.sql.functions as F

# Definir las variables categóricas
categorias = #RELLENAR

# Crear una tabla de contingencia
tabla_contingencia = df.groupby(categorias).agg(F.count('*').alias('count')) \
    .groupBy(categorias[:]).pivot(categorias[-1]).agg(F.first('count')).fillna(0)

tabla_contingencia.show(2)

# Crear un mapa de calor
sns.heatmap(tabla_contingencia.toPandas().set_index(categorias[:]), cmap='coolwarm')

En el mapa de calor se pueden observar las correlaciones para cada una de las combinaciones de las variables categóricas y la variable de salida. Se aprecia una clara correlación para la secuencia 2-2-3-1-1-0 (la representación no es buena, no tiene porque verse la secuencia), es decir, **Hombre - Colesterol Muy Elevado - Glucosa Normal - Fumar - Beber - Sedentario**. En la tabla de contingencia se pueden analizar estos valores más profundamente. Esta combinación ofrece información relevante e indica la correlación entre estas categorías. Se tendría que estudiar en más profundidad, pero ya es una pista de una posible tendencia.

En suma, a parte de la combinación recién observada, nos hemos podido hacer una idea de que, en general, las variables no están relacionadas y que todas son importantes para el estudio. Además, se han podido detallar las relaciones entre los atributos y la influencia de estos sobre la variable objetivo.

### Técnicas de clustering

Ahora vamos a implementar técnicas de clustering para agrupar datos similares y descubrir patrones ocultos. Esto lo implementaremos sobre variables numéricas y realizaremos también técnicas de visualización.

No obstante, antes de comenzar con esta sección, vamos a obtener los conjuntos de datos numéricos y categóricos actualizados (después de la detección de outliers). Trabajaremos solo sobre variables numéricas. Una opción sería aplicar una técnica de *One-Hot Encoding*, por ejemplo, sobre las categóricas. Sin embargo, ahora solo queremos saber implementar el K-Means en pySpark, luego nos quedamos con las numéricas.

In [ ]:
# Dataset numérico
df_num_clean = df_clean.select("age", "height", "weight", "ap_hi", "ap_lo", "cardio")

# Dataset categórico
df_cat_clean = df_clean.drop("age", "height", "weight", "ap_hi", "ap_lo")

#### K-Means

KMeans es un algoritmo de clustering que tiene como objetivo particionar un conjunto de datos en k grupos (clusters), donde cada punto de datos pertenece a un solo cluster. Se aplica sobre variables numéricas. En este caso, para establecer el número k de clusters, aplicaremos gráficamente la regla del codo (se escoge el número k a partir del cual aprecia estabilidad). Para aplicar el algoritmo, se tienen que seguir los siguientes pasos:

1. Crear un VectorAssembler para unir todas las columnas de características en un solo vector denominado "features".
2. Crear un StandardScaler para escalar las características de modo que tengan una media cero y una desviación estándar de uno. Esto es importante para que las características se encuentren en la misma escala y tengan la misma importancia en el análisis de clustering.
3. Crear un modelo de KMeans con una semilla aleatoria establecida.
4. Crear un pipeline que une el VectorAssembler, el StandardScaler y el modelo de KMeans.
5. Mostrar las predicciones y hacer las gráficas correspondientes por pares.

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import StandardScaler
from pyspark.ml import Pipeline
import numpy as np

# REGLA DEL CODO
# Variables numéricas
cols = ['age', 'height', 'weight', 'ap_hi', 'ap_lo']

# Crear un VectorAssembler y outputCol="features"
assembler = #RELLENAR

# Crear un StandardScaler
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=True)

# Crear un KMeans
kmeans = KMeans(seed=1)

# Crear un pipeline con las componentes anteriores.
pipeline = #RELLENAR

# Ajustar el modelo con diferentes valores de K
cost = np.zeros(16)
for k in range(2,16):
    kmeans.setK(k)
    model = pipeline.fit(df_num_clean)
    cost[k] = model.stages[-1].summary.trainingCost

# Graficar la curva del codo
fig, ax = plt.subplots(1,1, figsize =(8,6))
ax.plot(range(2,16),cost[2:16], marker = "o")
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Costo")
plt.show()

Escogemos k = 5 clusters. La curva se comienza a suavizar ligeramente a partir de este punto. Tampoco es conveniente coger un número exagerado de clusters, por ello, escogemos 5.

In [ ]:
# K-MEANS
# Unimos todas las variables numéricas en un vector
assembler = VectorAssembler(
    inputCols=cols,
    outputCol="features")

# Escalamos las variables numéricas
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withStd=True,
    withMean=True)

# Entrenamos el modelo KMeans con 5 clusters y semilla 1
kmeans = #RELLENAR

# Creamos el pipeline
pipeline = #RELLENAR

# Ajustamos el pipeline al dataframe
model = #RELLENAR

# Obtenemos las predicciones
predictions = #RELLENAR

# Mostramos las predicciones
predictions.#RELLENAR

Quiero mirar cuantas instancias hay en cada cluster.

In [ ]:
# Para ello, contamos agrupando y ordenando por predicition
contador = #RELLENAR
contador.show()

In [ ]:
Test.assertEquals(set(contador.select('count').rdd.flatMap(lambda x: x).collect()), set([5294, 6614, 9585, 15676, 16286]), 'Clusters incorrecto')

In [ ]:
from matplotlib.colors import ListedColormap

# Definimos las características que queremos graficar
features = ['age', 'height', 'weight', 'ap_hi', 'ap_lo']

# Convertir DataFrame a array de características
features_array = np.array(predictions.select('age', 'height', 'weight', 'ap_hi', 'ap_lo').collect())

# Obtenemos las etiquetas de los clusters asignados a cada punto de datos
cluster_labels = predictions.select("prediction").rdd.flatMap(lambda x: x).collect()
colors = ListedColormap(['red', 'green','blue','yellow','orange'])

# Creamos todas las posibles combinaciones de pares de características
pairs = [(i,j) for i in range(len(features)) for j in range(i+1, len(features))]

# Creamos una figura con subplots de 2x5
fig, axs = plt.subplots(2, 5, figsize=(20,8))

# Iteramos sobre todas las combinaciones de pares de características
for idx, (i,j) in enumerate(pairs):
    # Obtenemos los valores de las características i y j
    x = features_array[:,i]
    y = features_array[:,j]

    # Obtenemos los valores de los clusters
    c = cluster_labels

    # Definimos la posición del subplot correspondiente
    row = idx // 5
    col = idx % 5

    # Graficamos el scatter plot correspondiente en el subplot correspondiente
    axs[row, col].scatter(x, y, c=c, cmap = colors)
    axs[row, col].set_xlabel(features[i])
    axs[row, col].set_ylabel(features[j])

# Mostramos la figura
plt.show()

Aquí se puede observar la distribución de los diferentes clusters en función de los pares de variables seleccionadas. Hay combinaciones para las que estos grupos se ven más diferenciados (*ap_hi* - *weight*, *ap_lo* - *weight* o *weight*-*height*), mientras que hay otras que menos (*height* - *age* o *ap_lo*-*age*).

### Técnicas de visualización de datos (PCA)

In [ ]:
from pyspark.ml.feature import PCA

# Se definen las columnas numéricas
num_cols = ['age', 'height', 'weight', 'ap_hi', 'ap_lo']

# Se crea un VectorAssembler para unir todas las características en un vector
assembler = #RELLENAR

# Se aplica el VectorAssembler a df_num_clean para crear un DataFrame con una nueva columna "features"
assembled_df = #RELLENAR

# Se crea un StandardScaler para escalar las características
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=True)

# Se aplica el StandardScaler a assembled_df para crear un nuevo DataFrame con una nueva columna "scaledFeatures"
scaled_df = scaler.fit(assembled_df).transform(assembled_df)

# Se crea un PCA para reducir la dimensionalidad de las características
pca = PCA(k=5, inputCol="scaledFeatures", outputCol="pcaFeatures")

# Se ajusta el modelo PCA a scaled_df para obtener el modelo entrenado
model = pca.fit(scaled_df)

# Se aplica el modelo PCA a scaled_df para crear un nuevo DataFrame con una nueva columna "pcaFeatures"
pca_df = model.transform(scaled_df).select("pcaFeatures")

In [ ]:
pca_df.show(3)

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType

# Definir una función UDF para extraer cada componente principal de la columna pcaFeatures
def get_pca(col, idx):
    return udf(lambda x: float(x[idx]), FloatType())(col)

# Obtener cada componente principal en una columna separada
for i in range(5):
    pca_df = pca_df.withColumn(f"PCA{i+1}", get_pca("pcaFeatures", i))

# Seleccionar solo las columnas de las componentes principales
pca_df = pca_df.select("PCA1", "PCA2", "PCA3", "PCA4", "PCA5")

In [ ]:
pca_df.show(3)

In [ ]:
# Obtener la matriz de componentes principales
pca_matrix = np.array(model.pc.toArray()).reshape(5,5)

# Seleccionar las columnas PCA1 y PCA2 y convertirlas a un Pandas DataFrame
pca_plot_data = pca_df.select("PCA1", "PCA2").toPandas()

# Graficar el diagrama de dispersión
fig, ax = plt.subplots()
ax.scatter(pca_plot_data['PCA1'], pca_plot_data['PCA2'])
ax.set_xlabel("PCA1")
ax.set_ylabel("PCA2")

# Agregar flechas que representen los vectores de las componentes principales
for i, (x, y) in enumerate(zip(model.pc.toArray()[:,0], model.pc.toArray()[:,1])):
    ax.arrow(0, 0, x, y, head_width=0.01, head_length=0.01, fc='k', ec='k')
    ax.annotate(f"PC{i+1}", xy=(x, y), xytext=(x+0.2, y+0.2), fontsize=12)

plt.show()

Si las dos flechas son pequeñas y muy parecidas, significa que ambas componentes principales tienen un efecto similar en la varianza total de los datos. En este caso, puede ser difícil distinguir entre la influencia de cada componente en los datos.

In [ ]:
# Obtener la varianza explicada por cada componente principal
explained_variance = model.#RELLENAR
print("Varianza explicada por cada componente principal:\n", explained_variance)

# Obtener el porcentaje de varianza explicada por cada componente principal
total_variance = sum(explained_variance)
explained_variance_ratio = [round((ev / total_variance)*100,2) for ev in explained_variance]
print("\nPorcentaje de varianza explicada por cada componente principal:\n", explained_variance_ratio)

In [ ]:
Test.assertEquals(explained_variance_ratio, [36.25, 25.52, 18.7, 13.25, 6.28], 'Clusters incorrecto')

La primera componente principal explica el 36.25% de la varianza total de los datos originales, la segunda componente principal explica el 25.52%, y así sucesivamente hasta la quinta componente principal, que explica el 6.28%. En resumen, estos números indican cuánta información útil se retiene al reducir la dimensionalidad de los datos mediante el análisis de componentes principales. En general, se considera que una buena reducción de la dimensionalidad debe retener al menos el 70-80% de la varianza total de los datos originales.

Vemos que con las tres primeras componentes principales ya cumpliríamos este objetivo. Por tanto, de las 5 variables numéricas que teníamos al principio, podríamos reducir la dimensionalidad a 3 componentes principales, es decir, 3 combinaciones de estos atributos que explicarían en torno al 80% de la varianza de los datos.

### Técnicas aprendizaje automático

Para este problema de clasificación implementaremos la Regresión Logística. Para ello, trabajaremos solo sobre las variables numéricas y propondremos algunas técnicas de visualización adicionales, como por ejemplo, el área bajo la curva ROC. Además, lo implementaremos de una forma diferente a la vista en la práctica de clasificación, con el objetivo de ver maneras distintas de diseñar este modelo.

In [ ]:
df_num_clean = df_num_clean.withColumn("cardio", when(df_num_clean["cardio"] == "1", 1).otherwise(0))

#### Entrenamiento del modelo y predicciones

In [ ]:
from pyspark.ml.feature import VectorIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Seleccionamos las columnas de características y la variable de salida
featuresCols = df_num_clean.columns
featuresCols.remove('cardio')

# Creamos un VectorAssembler para unir todas las características en un vector
vectorAssembler = VectorAssembler(inputCols=featuresCols, outputCol="rawFeatures")

# Identificamos las características categóricas y las indexamos
vectorIndexer = VectorIndexer(inputCol="rawFeatures", outputCol="features", maxCategories=4)

# Creamos el modelo de regresión logística
lr = LogisticRegression(featuresCol="features", labelCol="cardio")

# Creamos el pipeline
pipeline = #RELLENAR

# Dividimos los datos en entrenamiento y prueba con semilla 100
(trainingData, testData) = #RELLENAR

# Entrenamos el modelo con los datos de entrenamiento
model = #RELLENAR)

#### Métricas de rendimiento

In [ ]:
# Hacemos las predicciones con los datos de entrenamiento
prediction_train = #RELLENAR

# Evaluamos el modelo
evaluator = BinaryClassificationEvaluator(labelCol="cardio", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
roc = evaluator.#RELLENAR

In [ ]:
Test.assertEquals(round(roc,2), 0.77, 'AUC incorrecto')

In [ ]:
# Hacemos las predicciones con los datos de prueba (realmente lo que nos interesa)
predictions = #RELLENAR

# Evaluamos el modelo
evaluator = BinaryClassificationEvaluator(labelCol="cardio", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
roc = evaluator.#RELLENAR

print("El área bajo la curva ROC es: {:.2f}".format(roc))

In [ ]:
Test.assertEquals(round(roc,2), 0.77, 'AUC incorrecto')

Un valor del área bajo la curva (AUC) de 0.77 indica que el modelo tiene una capacidad razonable para discriminar entre las clases positiva y negativa. En general, un valor de AUC entre 0.7 y 0.8 se considera aceptable.

Ahora calcularemos el RMSE. Lo hacemos solo sobre el conjunto test que es el que realmente nos interesa.

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

# Creamos el evaluador para la regresión
evaluator = RegressionEvaluator(labelCol="cardio", predictionCol="prediction", metricName="rmse")

# Evaluamos el modelo
rmse = evaluator.#RELLENAR

# Imprimimos el resultado
print("El RMSE en test es: {:.2f}".format(rmse))

In [ ]:
# Evaluamos el modelo
rmse = evaluator.#RELLENAR

# Imprimimos el resultado
print("El RMSE en test es: {:.2f}".format(rmse))

In [ ]:
Test.assertEquals(round(rmse,2), 0.54, 'RMSE incorrecto')

Cuanto más pequeño sea el RMSE, mejor será el modelo para ajustar los datos. Por lo tanto, un RMSE de 0.54 puede ser una buena métrica de evaluación del modelo dependiendo del contexto del problema y la magnitud de las variables objetivo.

#### Visualización

Observamos la estructura del dataframe *predictions*. Mostramos las 2 primeras líneas.

In [ ]:
predictions.show(2)

In [ ]:
from sklearn.metrics import roc_curve, auc

# Obtener los valores de probabilidad y las etiquetas reales
probs = predictions.select('probability').rdd.map(lambda x: x[0][1]).collect()
labels = predictions.select('cardio').rdd.map(lambda x: x[0]).collect()

# Calcular la curva ROC
fpr, tpr, thresholds = roc_curve(labels, probs)

# Calcular el área bajo la curva ROC
roc_auc = auc(fpr, tpr)

# Graficar la curva ROC
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic')
plt.legend(loc="lower right")
plt.show()

#### Validación cruzada


Se toma una lista de combinaciones de parámetros y una medida de evaluación y él automáticamente busca la mejor combinación mediante validación cruzada. Aplicamos la validación cruzada vista en las prácticas.

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Usamos el VectorAssembler para agregar la columna "features" a los datos
data = assembler.transform(trainingData).withColumnRenamed("cardio", "label").select("features", "label")

# Creamos una cuadrícula de parámetros a ajustar
paramGrid = (ParamGridBuilder()
             .addGrid(lr.regParam, [0.1, 0.01])
             .addGrid(lr.elasticNetParam, [0.0, 0.1])
             .build())

# Usamos el evaluador de BinaryClassification por defecto para la validación cruzada
eva = #RELLENAR

# Creamos un objeto CrossValidator para la validación cruzada
crossval = CrossValidator(estimator=lr,
                          estimatorParamMaps=paramGrid,
                          evaluator=eva,
                          numFolds=5)

# Ajustamos el modelo con los datos de entrenamiento usando la validación cruzada
cvModel = crossval.#RELLENAR

In [ ]:
# Realizamos la predicción en los datos de prueba
predictions_cv = cvModel.#RELLENAR

In [ ]:
# Renombramos la columna 'cardio' a 'label' en los datos de prueba
predictions_cv = predictions_cv.withColumnRenamed("cardio", "label")

# Evaluamos el modelo en los datos de prueba
areaUnderROC_cv = eva.evaluate(predictions_cv)

print("Area Under ROC = {}".format(areaUnderROC_cv))

Observamos que mejora ligeramente (o se mantiene sin mejora) la métrica obtenida.

### Conclusiones

El objetivo de esta tarea ha sido el de realizar un estudio completo de un dataset de gran tamaño. La selección de este problema ha estado muy ligada al número de instancias del dataset, ya que al contener 70.000 ejemplos me pareción adecuado para la asignatura.

Por otra parte, he tratado de ampliar los contenidos vistos en las otras prácticas, añadiendo secciones como:

- Análisis de datos
- Relación entre variables
- Técnicas de clustering o agrupación de datos
- Reducción de la dimensionalidad
- Modelos de clasificación
- Visualización de la información

In [ ]:
from IPython.display import display

In [ ]:
# Se desactiva primero la acción de %display latex, si estuviera activa
%display default

In [ ]:
%%html
<style>
h1{text-align: center; color: rgb(185,74,72);}
h2{text-align: center; color: rgb(0,102,0); padding: 0.25em 0;
   border: 2px solid rgb(0,191,0); border-width: 2px 0;}
h3{border-bottom: 2px solid rgb(153,153,153);}
h4{color: rgb(58,135,173); font-size: 115%!important;
   font-weight: bold!important;}
.text_cell_render{font-family: "Trebuchet MS",Geneva,sans-serif;
                  font-size: 110%; line-height: 1.5;}
.MathJax_Display{margin: 0.5em 0;}
</style>